# Solving models in ModelFlow

A ModelFlow `model` instance is a *callable*: applied to a Pandas DataFrame it solves the model
over a time sample and returns a new DataFrame with the solution.

```python
result = mmodel(df, '2021q1', '2035q4')            # solve df over the sample
result = mmodel(df, '2021q1', '2035q4', silent=0)  # ... with progress output
```

Behind the call, ModelFlow translates the model's equations into Python source code
(optionally compiled with numba), determines a solve order from the dependency graph,
and runs a solver appropriate for the model's structure. Three structural properties decide
almost everything:

* **Leads.** If any equation refers to a *future* value (`X(+1)`), the periods cannot be
  solved one at a time: all periods are simultaneous and must be *stacked* into one big
  system.
* **Simultaneity.** If the current-period dependency graph is acyclic (*topological* model),
  a single sweep through the equations in the right order solves a period exactly.
  Otherwise the simultaneous block must be solved iteratively (Gauss-Seidel or Newton).
* **Equation form.** Normalized or implicit — see the next section. Implicit equations
  *require* a Newton solver.

## Equation types: normalized and implicit

**Normalized** equations have the endogenous variable alone on the left-hand side:

$$y_t = F(y_t, x_t, \dots)$$

```
FRML <> y = 0.5 * y(-1) + 0.2 * x $
```

A normalized equation *assigns* its variable, so a Gauss-Seidel sweep (or a topological
sweep for acyclic models) can solve it directly: evaluate the right-hand side, store the
result in `y`, repeat until convergence.

**Implicit** (un-normalized, residual-form) equations cannot — or should not — be solved
for their unknown analytically. They are written as *residual equations*: the equation
variable is a residual `..___RES`, and the actual unknown is declared with `endo=` in the
FRML name:

$$y^{RES}_t = G(y_t, x_t, \dots) \quad\text{solve for } y_t \text{ such that } y^{RES}_t = 0$$

```
FRML <endo=y> y___RES = y**2 + log(y) - x $
```

No sweep can assign `y` here — only a Newton solver can, by moving the declared unknown
until the residual is zero. The helper `modelmanipulation.un_normalize_expression` (and the
`normal(..., implicit=True)` pipeline in `modelnormalize`) rewrites normalized equations to
this form when a model is deliberately kept implicit.

The model records which world it lives in:

| Property | Meaning |
|---|---|
| `mmodel.normalized` | no `___RES` equations — every solver is available |
| `mmodel.implicit` | *all* equations are residual-form |
| `mmodel.hybrid` | a mix of normalized and implicit equations |

**Consequence for solver choice:** `xgenr`, `sim` and `sim1d` only work on fully normalized
models. As soon as one implicit equation is present, a Newton solver is required — the
legacy `newton_un_normalized` / `newtonstack_un_normalized`, or the next-generation
`newton_ng` / `newtonstack_ng`, which are *unified*: a per-equation mask selects the residual
convention (`G(y)` on residual rows, `F(y) - y` on normalized rows), so normalized, implicit
and hybrid models are all handled by the same solver.

## Automatic solver dispatch

When the model is called without an explicit `solver=`, `model.__call__` inspects the
structure and picks a default:

| Model structure | Dispatched solver |
|---|---|
| leads (`maxlead >= 1`), normalized | `newtonstack` |
| leads, implicit/hybrid equations | `newtonstack_un_normalized` |
| no leads, normalized, current-period graph acyclic (`istopo`) | `xgenr` |
| no leads, normalized, simultaneous | `sim` |
| no leads, implicit/hybrid equations | `newton_un_normalized` |

Any solver can be forced with the `solver=` option, including the next-generation family by
its `_ng` name:

```python
result = mmodel(df, solver='newton_ng')
result = mmodel(df, solver='newtonstack_ng', split=1)
```

*The plan is to let the dispatch table select the `_ng` solvers once they are fully proven;
for now the legacy solvers remain the default and the `_ng` family is opt-in.*

### Options are sticky

`model.__call__` remembers the options of the previous call and merges them into the next
one (`{**self.oldkwargs, **kwargs}`). Explicitly passed options always win, but an option
you *stop mentioning* keeps its old value. This is convenient in interactive work — set
`ljit=1` once — but surprising when switching solvers: a `jacobian='fd'` left over from a
`newton_fbmin_ng` call will leak into a later `newtonstack_fbmin_ng` call. Reset with:

```python
result = mmodel(df, reset_options=True, ...)   # forget all remembered options
```
When debugging odd solver behaviour, inspect `mmodel.oldkwargs` first.

## Common options

These options work across (nearly) all solvers. Positional: `mmodel(databank, start, end, ...)`.

| Option | Default | Meaning |
|---|---|---|
| `start`, `end` | current sample | first / last period to solve (also settable via `mmodel.smpl` / `set_smpl`) |
| `solver` | structural guess | select the solver by name (see dispatch table) |
| `silent` | `1` | suppress progress output |
| `stats` | `0` | print timing / floating-point-operation statistics after the solve |
| `timeit` / `timeon` | `0` | fine-grained timers for the phases of the solve (`timeit` in the `_ng` and Newton solvers, `timeon` in legacy `sim`) |
| `max_iterations` | solver-specific | iteration limit (Gauss-Seidel sweeps or Newton steps) |
| `alfa` | `1.0` | damping of the Gauss-Seidel update |
| `conv` | `'*'` | variable pattern the convergence test is applied to (Gauss-Seidel) |
| `absconv` | `0.01` | only variables with `abs(value) >= absconv` enter the convergence test |
| `relconv` | `1e-7` | converged when every tested variable changes relatively less than this |
| `ljit` | `False` | compile the model evaluation with numba (first call pays compilation, later calls are fast; compiled code is cached on disk via a transpiled file in `modelsource/`) |
| `stringjit` | `False` | jit from an in-memory string instead of the transpiled file (no on-disk cache) |
| `transpile_reset` | `False` | force regeneration of the transpiled solver file |
| `chunk` | `30` | equations per generated function chunk (keeps numba compile time sane) |
| `samedata` | | assume the databank layout is unchanged, reuse the compiled evaluator |
| `init` | `False` | start each period's iteration from the previous period's solution |
| `keep` | `''` | store the solution under this label in `mmodel.keep_solutions` (for comparison plots) |
| `keep_variables` | `''` | restrict what `keep` stores |
| `save` | model default | update `basedf`/`lastdf` bookkeeping |
| `ldumpvar`, `dumpvar`, `dumpwith`, `dumpdecimal` | `False`, `'*'`, ... | record per-iteration values of selected variables in `mmodel.dumpdf` (debugging) |
| `reset_options` | `False` | forget sticky options from earlier calls |
| `keep_residual` | `False` | (`_ng` only) store the last-iteration equation residuals as `mmodel.ng_residual` |
| `progressbar` | `False` | tqdm progress bar (legacy `sim`) |

## The legacy solvers (`modelclass.py`)

### `xgenr` — topological sweep
For *normalized* models whose current-period graph is acyclic. Each period is solved by one
exact pass through the equations in topological order: no iteration, no convergence test,
no Jacobian. This is what the dispatcher picks for definition-style models.

### `sim` — damped Gauss-Seidel
The workhorse for backward-looking simultaneous *normalized* models. Iterates sweeps over
the simultaneous core until the relative-change convergence test passes; the recursive
prolog/epilog parts of the model are evaluated once before/after the core iteration.
Specific options: `alfa` (damping), `first_test` (first iteration to test convergence),
`conv`/`absconv`/`relconv`.

### `sim1d` — Gauss-Seidel on a stuffed 1-D array
Same algorithm as `sim`, but the generated code addresses a one-period 1-D array with
compile-time offsets instead of a 2-D (period x variable) array — better data locality,
usually the fastest Gauss-Seidel variant for large models. Extra option: `cache` for the
numba on-disk cache.

### `res` — one evaluation pass
Not an equation *solver*: evaluates every equation once per period against the input data
(lagged endogenous read from the input, not from computed results) and returns the result.
Used for residual checking and internally to evaluate derivative models.

### `newton` / `newton_un_normalized` — per-period Newton
Newton-Raphson on each period's simultaneous core: the Jacobian is built symbolically or
numerically by `modelnewton.newton_diff`, sparse-LU-factorized per period, and reused
across periods and iterations until refreshed. `newton_un_normalized` is the variant for
implicit (residual-form) equations — required as soon as the model is not fully normalized.

| Option | Default | Meaning |
|---|---|---|
| `nonlin` | `False` | refresh the Jacobian every `nonlin` iterations (for nonlinear models) |
| `newton_reset` | `False` | force a rebuild of the Jacobian machinery |
| `newtonalfa`, `newtonnodamp` | `1.0`, `0` | damping of the Newton step for the first `newtonnodamp` iterations |
| `lnjit` | `False` | jit the *derivative* model |
| `forcenum` | `False` | force numerical (finite-difference) derivatives instead of symbolic |

### `newtonstack` / `newtonstack_un_normalized` — stacked Newton
For models with leads: all periods are stacked into one system of
`n_variables * n_periods` unknowns and solved by Newton with one big sparse LU.
The dispatcher's choice for forward-looking (model-consistent expectations) models;
the `_un_normalized` variant when implicit equations are present.

| Option | Default | Meaning |
|---|---|---|
| `nonlin`, `newton_reset`, `newtonalfa`, `newtonnodamp` | | as for `newton` |
| `nljit` | `0` | jit the derivative model |
| `nchunk` | `None` (30 when jitted) | chunk size for the derivative model's generated code |
| `forcenum` | | numerical derivatives |

*The stacked LU is factorized with SuperLU using the band-preserving `'NATURAL'` column
ordering by default — on FRB/US MCE this is ~10x faster than fill-reducing orderings,
because the stacked matrix is block-banded by construction.*

## The next-generation solvers (`modelsolver_ng.py`)

The `_ng` family reimplements the solvers on a shared template (`SolverBase`): one common
setup (sample, data preparation, code generation via a single `makelos`), a solver-specific
`prepare`/`iterate`, and a common teardown (result frame, statistics, residual recording).
Nothing shadows the legacy solvers — every entry point carries the `_ng` suffix and is
selected with `solver='<name>_ng'`.

The `_ng` Newton solvers are *unified*: normalized, implicit and hybrid models are all
handled by the same solver through a per-equation residual mask, so there are no separate
`_un_normalized` variants.

All common options above apply; the per-solver specifics follow.

### `xgenr_ng`, `sim_ng`, `sim1d_ng`, `res_ng`
Direct ports of their legacy counterparts (topological sweep, Gauss-Seidel 2-D / 1-D,
one-pass evaluation) on the shared skeleton — normalized models only, like the originals.
`res_ng` regenerates and byte-compares the transpiled jit file, so a stale compiled
evaluator is refreshed automatically.

### `newton_ng` — unified per-period Newton
Covers normalized **and** implicit equations in one solver, replacing the
`newton` / `newton_un_normalized` pair.

| Option | Default | Meaning |
|---|---|---|
| `nonlin`, `newton_reset`, `newtonalfa`, `newtonnodamp` | | as legacy `newton` |
| `lnjit` | `False` | jit the derivative model |
| `forcenum` | `False` | numerical derivatives |
| `diff_ng` | `0` | evaluate the derivative model through `res_ng` instead of legacy `res` |
| `permc_spec` | `None` (COLAMD) | SuperLU column ordering for the per-period LU |
| `factorizer` | `None` | `'umfpack'`: cvxopt/UMFPACK with the symbolic analysis cached and reused across periods and `nonlin` refreshes — pays off when nonlinear models refactorize often |
| `split` | `0` | graph-based pro/core/epi split of the *current-period* graph: Newton (and the per-period LU) only on variables in current-period cycles plus all `___RES` equations; prolog swept before, epilog after each period. Useful for `straight=True` and implicit/hybrid models, where `coreorder` is unavailable or blind to the coupling through an implicit equation's declared unknown |

### `newton_fbmin_ng` — per-period Newton on the minimal feedback set
Decomposes each period's core into a DAG part (solved exactly by topological sweeps) and a
*minimal feedback vertex set* that Newton iterates. Very fast when the feedback set is small.

| Option | Default | Meaning |
|---|---|---|
| `jacobian` | `'fd'` | `'fd'`: dense finite differences of the reduced residual; `'direct'`: `newton_diff` on the fb equations only (degenerates to fixed point when there are no direct fb-fb dependencies, which is the empirical norm); `'gauss'`: damped fixed point, no Jacobian (robustness fallback) |
| `nonlin` | `max(n_fb, 2)` | refresh interval (break-even is about the feedback set size) |

### `newtonstack_ng` — unified stacked Newton (alias `newtonstack_implicit_ng`)
The stacked solver for models with leads, normalized and implicit equations alike.

| Option | Default | Meaning |
|---|---|---|
| `nonlin`, `newton_reset`, `newtonalfa`, `newtonnodamp` | | as legacy `newtonstack` |
| `nljit` | `0` | jit the derivative model (measured *slower* on FRB/US MCE — the derivative model is evaluated once per Jacobian build, so there is nothing for numba to amortize; leave off unless `nonlin` refreshes are frequent) |
| `nchunk` | `chunk` when jitted | derivative-model chunk size |
| `diff_ng` | `0` | evaluate the derivative model through `res_ng` |
| `permc_spec` | `'NATURAL'` | SuperLU ordering of the stacked LU; the band-preserving default is ~10x faster than COLAMD on block-banded stacked systems (13s to 1.2s on FRB/US MCE). Pass `'COLAMD'` to compare; never use `'MMD_AT_PLUS_A'` here (fill explosion) |
| `factorizer` | `None` | `'umfpack'`: symbolic analysis reused across rebuilds; measured a wash vs SuperLU/NATURAL on FRB/US (faster factorization, slower backsolves) |
| `split` | `0` | **stacked pro/core/epi split**: Newton (and the LU) only on variables involved in collapsed-graph cycles (variables depending on both their own lag *and* lead are promoted to core — a true stacked cycle); upstream *prolog* variables swept exactly once before, pure reporting *epilog* variables once after convergence. Handles implicit equations (`___RES` always core, dependencies on declared unknowns mapped to their residual equation). FRB/US MCE: 234 core / 29 prolog / 22 epilog, LU 1.23s to 0.71s |

The Jacobian build (derivative evaluation + sparse assembly + LU) is reported as a separate
`Jacobian build time` stats line, since it runs outside the simulation-time measurement.

### `newtonstack_fbmin_ng` — stacked Newton on the stacked feedback set
The feedback-set idea applied to the *stacked* (period x variable) graph, so it also covers
leads. Non-feedback equations are solved exactly by topological sweeps that may run across
periods; Newton iterates only the stacked feedback unknowns.

Its niche: **lead-only recursive models** (empty feedback set — solved exactly by a single
stacked sweep, no Jacobian at all, which no other solver can do) and small models with small
stacked feedback sets. For large forward-looking models the reduced Jacobian's *dense*
`n_fb x n_fb` LU loses to `newtonstack_ng` + `split` — the decomposition-size print tells you
immediately which regime you are in. Inspect the feedback set via `mmodel.ng_stackfbmin_fb`.

| Option | Default | Meaning |
|---|---|---|
| `jacobian` | `'stack'` | `'stack'`: Schur complement of the stacked Jacobian (exact stacked Newton step on the reduced system); `'fd'`: finite differences (`n_fb + 1` full-span sweeps per build — only for small fb sets); `'gauss'`: damped fixed point |
| `keep_residual` | `False` | store the feedback residual as `mmodel.ng_stackfbmin_residual` |

SCCs larger than `LARGE_SCC_COLLAPSE` (1000 nodes) use the feedback set of the collapsed
variable-level graph — sufficient and fast where the exact cycle-by-cycle heuristic would hang.

## Choosing a solver, and making it fast

1. **Trust the dispatch first.** The structural guess (`xgenr` / `sim` / `newton*` /
   `newtonstack*`) is correct for the model class; the `_ng` variants are drop-in
   replacements to opt into. Remember: any implicit equation rules out the sweep and
   Gauss-Seidel solvers — Newton is mandatory.
2. **Jit the model** (`ljit=1`) when you solve more than a handful of times. The first call
   pays numba compilation; the transpiled file in `modelsource/` caches the compiled code
   across sessions.
3. **Reuse the Jacobian** in experiment loops: `newton_reset=0` keeps the factorization
   from the previous solve, and Newton converges fine from a slightly stale Jacobian for
   moderate shocks (`nonlin` still refreshes mid-solve if a hard shock needs it).
4. **`split=1`** on `newtonstack_ng` removes the pure-reporting and pure-upstream variables
   from the stacked factorization at no cost in accuracy.
5. Leave the derivative model un-jitted (`nljit=0`) unless `nonlin` refreshes are frequent.

As a calibration point, the FRB/US model-consistent-expectations shock loop
(285 equations, 124 quarters, stacked system of ~35,000 unknowns):

| Configuration | Per solve |
|---|---|
| `newtonstack_ng`, COLAMD ordering, rebuild every solve | ~14.5 s |
| + `permc_spec='NATURAL'` (now the default) | ~3.0 s |
| + `split=1`, jit warm | ~2.1 s |
| + `newton_reset=0` (reuse factorization across shocks) | **~0.1-0.2 s** |

## Examples

*(to be added — one worked example per solver family: a topological model for `xgenr`, a
backward-looking simultaneous model for `sim`/`newton_ng`, an implicit model showing why
Newton is required, and a forward-looking model for `newtonstack_ng` with and without
`split`.)*